In [0]:
from pyspark.sql import functions as F

path = "/Volumes/workspace/sncf_bronze/landing/stops.txt"

df_stops_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(path)
)

display(df_stops_raw)

stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station
StopArea:OCE71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,1,null
StopPoint:OCETGV INOUI-71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,0,StopArea:OCE71043075
StopArea:OCE71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,1,null
StopPoint:OCETGV INOUI-71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,0,StopArea:OCE71718010
StopArea:OCE71793000,GIRONA,null,41.97937100,2.816957000,null,null,1,null
StopPoint:OCETGV INOUI-71793000,GIRONA,null,41.97937100,2.816957000,null,null,0,StopArea:OCE71793000
StopArea:OCE71793150,Portbou,null,42.42470100,3.158034000,null,null,1,null
StopPoint:OCECar TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150
StopPoint:OCETrain TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150
StopArea:OCE80021402,Augsburg Hbf,null,48.36540000,10.88600000,null,null,1,null


In [0]:
df_stops_raw.printSchema()

print("Nombre de lignes :", df_stops_raw.count())
print("Nombre de colonnes :", len(df_stops_raw.columns))

root
 |-- stop_id: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- stop_desc: string (nullable = true)
 |-- stop_lat: string (nullable = true)
 |-- stop_lon: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- stop_url: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- parent_station: string (nullable = true)

Nombre de lignes : 8678
Nombre de colonnes : 9


In [0]:
from pyspark.sql import functions as F

# Comptage des valeurs nulles par colonne
null_counts = df_stops_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_stops_raw.columns
])

display(null_counts)

stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station
0,0,8678,0,0,8678,8678,0,3398


In [0]:
display(
    df_stops_raw
    .groupBy("stop_id")
    .count()
    .filter(F.col("count") > 1)
)

stop_id,count


In [0]:
display(
    df_stops_raw.select(
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon",
        "location_type",
        "parent_station"
    ).limit(20)
)

In [0]:
#ajout des métadonnées d’ingestion :
from pyspark.sql import functions as F

df_stops_bronze = (
    df_stops_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_system", F.lit("SNCF_GTFS"))
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn(
        "_batch_id",
        F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")
    )
)

In [0]:
display(df_stops_bronze.limit(20))

stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,_ingestion_timestamp,_source_system,_source_file,_batch_id
StopArea:OCE71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,1,null,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopPoint:OCETGV INOUI-71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,0,StopArea:OCE71043075,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopArea:OCE71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,1,null,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopPoint:OCETGV INOUI-71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,0,StopArea:OCE71718010,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopArea:OCE71793000,GIRONA,null,41.97937100,2.816957000,null,null,1,null,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopPoint:OCETGV INOUI-71793000,GIRONA,null,41.97937100,2.816957000,null,null,0,StopArea:OCE71793000,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopArea:OCE71793150,Portbou,null,42.42470100,3.158034000,null,null,1,null,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopPoint:OCECar TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopPoint:OCETrain TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310
StopArea:OCE80021402,Augsburg Hbf,null,48.36540000,10.88600000,null,null,1,null,2026-09-04T09:13:10.116Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091310


In [0]:
(
    df_stops_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.sncf_bronze.stops")
)

In [0]:
df_bronze = spark.table("workspace.sncf_bronze.stops")

display(df_bronze.limit(20))

stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,_ingestion_timestamp,_source_system,_source_file,_batch_id
StopArea:OCE71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETGV INOUI-71043075,FIGUERES-VILAFANT,null,42.26458100,2.943028000,null,null,0,StopArea:OCE71043075,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETGV INOUI-71718010,Barcelone-Sants,null,41.37896100,2.139834000,null,null,0,StopArea:OCE71718010,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE71793000,GIRONA,null,41.97937100,2.816957000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETGV INOUI-71793000,GIRONA,null,41.97937100,2.816957000,null,null,0,StopArea:OCE71793000,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE71793150,Portbou,null,42.42470100,3.158034000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCECar TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopPoint:OCETrain TER-71793150,Portbou,null,42.42470100,3.158034000,null,null,0,StopArea:OCE71793150,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
StopArea:OCE80021402,Augsburg Hbf,null,48.36540000,10.88600000,null,null,1,null,2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447


In [0]:
print("Source :", df_stops_raw.count())
print("Bronze :", df_bronze.count())

Source : 8678
Bronze : 8678


In [0]:
display(
    df_bronze.select(
        "_ingestion_timestamp",
        "_source_system",
        "_source_file",
        "_batch_id"
    ).limit(10)
)

_ingestion_timestamp,_source_system,_source_file,_batch_id
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447
2026-09-04T09:14:47.253Z,SNCF_GTFS,dbfs:/Volumes/workspace/sncf_bronze/landing/stops.txt,20260904091447


In [0]:
#ingester tous les fichies
from pyspark.sql import functions as F

def ingest_gtfs_file(file_name):

    path = f"/Volumes/workspace/sncf_bronze/landing/{file_name}.txt"

    df_raw = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(path)
        .select("*", "_metadata.file_path")
    )

    df_bronze = (
        df_raw
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_system", F.lit("SNCF_GTFS"))
        .withColumnRenamed("file_path", "_source_file")
        .withColumn(
            "_batch_id",
            F.date_format(F.current_timestamp(), "yyyyMMddHHmmss")
        )
    )

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.sncf_bronze.{file_name}")
    )

    print(f"{file_name} ingéré")

In [0]:
gtfs_files = [
    "agency",
 "calendar_dates",
 "feed_info",
 "routes",
 "stop_times",
    "trips",
    "transfers"
   
]

for file_name in gtfs_files:
    ingest_gtfs_file(file_name)

agency ingéré
calendar_dates ingéré
feed_info ingéré
routes ingéré
stop_times ingéré
trips ingéré
transfers ingéré
